In [ ]:
from miditok import REMI, TokenizerConfig
from pathlib import Path
import numpy as np
import keras

### Define model name, tokenizer name, and timesteps

In [ ]:
model_name = "model.keras"
tokenizer_name = "tokenizer.json"
timesteps = 50

### Setup dataset

In [ ]:
from helpers import create_random_midi_folder

# Create a random dataset under the folders midis_train, midis_test, midis_val
create_random_midi_folder([(10, "midis_train"), (2, "midis_test"), (2, "midis_val")])

### Create a Tokenizer

In [ ]:
tokenizer = REMI(TokenizerConfig(use_programs=True))
paths_midis = list(Path("midis_train").glob('**/*.mid'))
paths_midis.extend(list(Path("midis_test").glob('**/*.mid')))
paths_midis.extend(list(Path("midis_val").glob('**/*.mid')))

tokenizer.train(
    vocab_size=2000,
    model="BPE",
    files_paths=paths_midis,
)
tokenizer.save(tokenizer_name)

### Load a tokenizer

In [ ]:
tokenizer = REMI(params=tokenizer_name)
paths_midis = list(Path("midis_train").glob('**/*.mid'))
paths_midis.extend(list(Path("midis_test").glob('**/*.mid')))
paths_midis.extend(list(Path("midis_val").glob('**/*.mid')))

### Setup training data

In [ ]:
from helpers import load_songs_random, setup_timestep_data

paths_train = list(Path("midis_train").glob('**/*.mid'))
paths_val = list(Path("midis_val").glob('**/*.mid'))

# Load in randomly selected songs
train_data = load_songs_random(tokenizer, paths_train, 10)
val_data = load_songs_random(tokenizer, paths_val, 2)

vocab_size = tokenizer.vocab_size

x_train, y_train = setup_timestep_data(train_data, timesteps)
x_val, y_val = setup_timestep_data(val_data, timesteps)

### Create a model

In [ ]:
model = keras.Sequential([
    keras.layers.Embedding(vocab_size, 256),
    keras.layers.LSTM(1024, return_sequences=True),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(1024),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(vocab_size, activation='softmax'),
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(timesteps, 1))
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=60, validation_data=(x_val, y_val))

In [ ]:
model.save(model_name)

In [ ]:
model = keras.models.load_model(model_name)

In [ ]:
generated_notes = []
x = tokenizer.encode("midis_train/Ashford, Emma Louise, He Leadeth Me, lZtxBhI5aIA.mid")
seed = np.array(x[0:50])
print(seed)

num_notes_to_generate = 10

for _ in range(num_notes_to_generate):
    seed = np.reshape(seed, (1, len(seed), 1))
    prediction = np.argmax(model.predict(seed, verbose=0))
    
    seed = np.append(seed, prediction)[len(seed)-50:]
    
    generated_notes.append(int(prediction))

print(generated_notes)

In [ ]:
tokenizer(generated_notes).dump_midi("generated.mid")

### Export this model to the expected file format of the GUI

In [ ]:
from helpers import create_model_folder

output_directory_name = "lstm_model"

create_model_folder(output_directory_name, model_name, tokenizer_name, timesteps, False)